In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [2]:
path = r"C:\Users\dell\Downloads\title.basics.tsv.gz"

# Gzip TSV file read karne ke liye
df = pd.read_csv(
    path, 
    sep='\t', 
    compression='gzip', 
    na_values='\\N', 
    low_memory=False
)

# Pehli 5 rows dekhne ke liye
df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [3]:
df.endYear.isnull().sum()

np.int64(12705294)

In [4]:
df.isnull().sum()

tconst                   0
titleType                0
primaryTitle            25
originalTitle           25
isAdult                  0
startYear          1485721
endYear           12705294
runtimeMinutes     8011043
genres              544902
dtype: int64

In [5]:
df.shape

(12868327, 9)

In [6]:
movies_df = df[df['titleType'] == 'movie']
print(movies_df.shape)

(759696, 9)


In [7]:
movies_df.isnull().sum()

tconst                 0
titleType              0
primaryTitle           3
originalTitle          3
isAdult                0
startYear         113740
endYear           759696
runtimeMinutes    271523
genres             77845
dtype: int64

In [8]:
movies_df = movies_df.drop(columns=['endYear'])

In [9]:
movies_df = movies_df.dropna(subset=['primaryTitle', 'originalTitle'])

In [10]:
movies_df['genres'] = movies_df['genres'].fillna('Unknown')

In [11]:
movies_df.isnull().sum()

tconst                 0
titleType              0
primaryTitle           0
originalTitle          0
isAdult                0
startYear         113740
runtimeMinutes    271522
genres                 0
dtype: int64

In [12]:
# 1. Missing release year wali rows drop karein (Bina release year movie compare nahi hogi)
movies_df = movies_df.dropna(subset=['startYear'])

# 2. Year ko integer type me convert karein
movies_df['startYear'] = movies_df['startYear'].astype(int)

# 3. runtimeMinutes ko numeric banayein aur missing values ko median se fill karein
movies_df['runtimeMinutes'] = pd.to_numeric(movies_df['runtimeMinutes'], errors='coerce')
movies_df['runtimeMinutes'] = movies_df['runtimeMinutes'].fillna(movies_df['runtimeMinutes'].median())

# 4. Final null values check karein
movies_df.isnull().sum()

tconst            0
titleType         0
primaryTitle      0
originalTitle     0
isAdult           0
startYear         0
runtimeMinutes    0
genres            0
dtype: int64

In [13]:
movies_df.shape

(645953, 8)

In [14]:
movies_df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,45.0,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,100.0,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,Bohemios,0,1905,100.0,Unknown
570,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,70.0,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,90.0,Drama


In [15]:
df_movies_recommneded = movies_df[['primaryTitle','originalTitle','genres','runtimeMinutes','startYear']]
df_movies_recommneded.head()

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
8,Miss Jerry,Miss Jerry,Romance,45.0,1894
144,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,"Documentary,News,Sport",100.0,1897
498,Bohemios,Bohemios,Unknown,100.0,1905
570,The Story of the Kelly Gang,The Story of the Kelly Gang,"Action,Adventure,Biography",70.0,1906
587,The Prodigal Son,L'enfant prodigue,Drama,90.0,1907


In [16]:
df_movies_recommneded['genres'] = df_movies_recommneded['genres'].map(lambda val: np.nan if val == " " else val)

In [17]:
df_movies_recommneded.isnull().sum()

primaryTitle      0
originalTitle     0
genres            0
runtimeMinutes    0
startYear         0
dtype: int64

In [18]:
df_movies_recommneded.head()

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
8,Miss Jerry,Miss Jerry,Romance,45.0,1894
144,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,"Documentary,News,Sport",100.0,1897
498,Bohemios,Bohemios,Unknown,100.0,1905
570,The Story of the Kelly Gang,The Story of the Kelly Gang,"Action,Adventure,Biography",70.0,1906
587,The Prodigal Son,L'enfant prodigue,Drama,90.0,1907


In [19]:
# Genres ko comma par split karke list banayein
df_movies_recommneded['genres'] = df_movies_recommneded['genres'].str.split(',')

# Check karein
df_movies_recommneded.head()

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
8,Miss Jerry,Miss Jerry,[Romance],45.0,1894
144,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,"[Documentary, News, Sport]",100.0,1897
498,Bohemios,Bohemios,[Unknown],100.0,1905
570,The Story of the Kelly Gang,The Story of the Kelly Gang,"[Action, Adventure, Biography]",70.0,1906
587,The Prodigal Son,L'enfant prodigue,[Drama],90.0,1907


In [20]:
# 1. Unknown genre wali movies ko filter out karein
df_movies_recommneded = df_movies_recommneded[
    df_movies_recommneded['genres'].apply(lambda x: 'Unknown' not in x)
].reset_index(drop=True)

# 2. Check karein kitni movies bachi hain
print("Bachi hui clean movies:", df_movies_recommneded.shape[0])
df_movies_recommneded.head()

Bachi hui clean movies: 575098


,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
0,Miss Jerry,Miss Jerry,[Romance],45.0,1894
1,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,"[Documentary, News, Sport]",100.0,1897
2,The Story of the Kelly Gang,The Story of the Kelly Gang,"[Action, Adventure, Biography]",70.0,1906
3,The Prodigal Son,L'enfant prodigue,[Drama],90.0,1907
4,Robbery Under Arms,Robbery Under Arms,[Drama],89.0,1907


In [21]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
df_movies_recommneded.index = df_movies_recommneded['primaryTitle']

In [23]:
encode  = MultiLabelBinarizer()
encode_data= encode.fit_transform(df_movies_recommneded['genres'])
df_encoded = pd.DataFrame(encode_data,index= df_movies_recommneded.index,columns= encode.classes_)
df_encoded.head()

,Action,Adult,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,...,Mystery,News,Reality-TV,Romance,Sci-Fi,Sport,Talk-Show,Thriller,War,Western
primaryTitle,,,,,,,,,,,,,,,,,,,,,
Miss Jerry,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
The Corbett-Fitzsimmons Fight,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,1,0,0,0,0
The Story of the Kelly Gang,1,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
The Prodigal Son,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
Robbery Under Arms,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df_encoded.index = df_encoded.index.str.lower().str.strip()

In [25]:
df_encoded.head()

,Action,Adult,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,...,Mystery,News,Reality-TV,Romance,Sci-Fi,Sport,Talk-Show,Thriller,War,Western
primaryTitle,,,,,,,,,,,,,,,,,,,,,
miss jerry,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
the corbett-fitzsimmons fight,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,1,0,0,0,0
the story of the kelly gang,1,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
the prodigal son,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
robbery under arms,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
def recommend(movie, n=5):
    movie_clean = movie.strip().lower()
    
    if movie_clean not in df_encoded.index:
        return "Movie Not Found!"
    
    # 1. Cosine similarity calculate karein
    result = cosine_similarity(df_encoded.loc[[movie_clean]], df_encoded).max(axis=0)
    result = pd.Series(result, index=df_encoded.index)
    
    # 2. Input movie ko hata kar duplicate titles handle karein
    result = result.drop(movie_clean, errors='ignore').groupby(level=0).max()
    
    # 3. Top N matches ke titles nikal kar list banayein
    top_titles = result[result > 0].nlargest(n).index.tolist()
    
    # 4. Filter karein aur index ko reset karein (drop=True se duplicate index hat jayega)
    output_df = df_movies_recommneded[
        df_movies_recommneded['primaryTitle'].str.lower().isin(top_titles)
    ].drop_duplicates(subset=['primaryTitle']).reset_index(drop=True)
    
    return output_df

In [27]:
recommend('miss jerry')

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
0,#69 Samskar Colony,#69 Samskar Colony,[Romance],126.0,2022
1,&Me,&Me,[Romance],88.0,2013
2,*YES I DO*,Yes I Do,[Romance],160.0,2024
3,#Single,#Single,[Romance],129.0,2025
4,...Smena nachinayetsya v shest,...Smena nachinayetsya v shest,[Romance],72.0,1958


In [28]:
recommend('miss jerry', n=20)

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
0,12 Hours,12 horas,"[Comedy, Drama, Music]",89.0,2001
1,1 2 3,1 2 3,[Comedy],94.0,2003
2,10th Class,10th Class,[Romance],150.0,2006
3,101 Proposals,101 ci qiu hun,[Romance],120.0,2013
4,0.1% World,Hao xiang qu ni de shi jie ai ni,[Romance],89.0,2022
5,10 Reasons Why Men Cheat,10 Reasons Why Men Cheat,[Romance],90.0,2022
6,#69 Samskar Colony,#69 Samskar Colony,[Romance],126.0,2022
7,1 Verabredung im Herbst,1 Verabredung im Herbst,[Romance],105.0,2021
8,&Me,&Me,[Romance],88.0,2013
9,*YES I DO*,Yes I Do,[Romance],160.0,2024


In [29]:
recommend('the story of the kelly gang')

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
0,Adrift,Touha zvaná Anada,[Drama],108.0,1971
1,"Aguirre, the Wrath of God","Aguirre, der Zorn Gottes","[Action, Adventure, Biography]",95.0,1972
2,100 Days,100 Days,"[Horror, Romance, Thriller]",161.0,1991
3,A Gamers Life,A Gamers Life,"[Action, Adventure, Biography]",146.0,2021
4,27 Guns,27 Guns,"[Action, Adventure, Biography]",89.0,2018


In [30]:
recommend('pk')

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
0,42nd Street,42nd Street,"[Comedy, Drama, Musical]",89.0,1933
1,40 años de novios,40 años de novios,"[Comedy, Drama, Musical]",88.0,1963
2,...Aur Pyaar Ho Gaya,...Aur Pyaar Ho Gaya,"[Comedy, Drama, Musical]",147.0,1997
3,36 China Town,36 China Town,"[Comedy, Drama, Musical]",140.0,2006
4,5 Seasons: The Movie,5 Seasons: The Movie,"[Comedy, Drama, Musical]",115.0,2013


In [31]:
recommend('joker')

,primaryTitle,originalTitle,genres,runtimeMinutes,startYear
0,...Aur Pyaar Ho Gaya,...Aur Pyaar Ho Gaya,"[Comedy, Drama, Musical]",147.0,1997
1,1/20,1/20,"[Comedy, Drama, Family]",90.0,2010
2,#Kaadal Kahaani,#Kaadal Kahaani,"[Comedy, Drama, Family]",93.0,2022
3,#Sequence,#Sequence,"[Comedy, Drama, Family]",63.0,2013
4,10 Kalpanakal,10 Kalpanakal,"[Crime, Mystery, Thriller]",125.0,2016
